In [ ]:
from __future__ import annotations
import operator
import re
from typing import TypedDict,List,Annotated,Literal
from pydantic import BaseModel,Field
from langgraph.graph import StateGraph,START,END
from langgraph.types import Send
from langchain_groq import ChatGroq
from langchain.messages import SystemMessage,HumanMessage
from dotenv import load_dotenv
load_dotenv()

#*******************Main Code*****************************************

class Task(BaseModel):
    id:int
    title:str
    goal:str=Field(description="What the reader should understand after this section")
    bullets:List[str]=Field(description="3-5 concrete subpoints",min_length=3,max_length=5)
    target_words:int=Field(description="Target word count",ge=120,le=450)
    section_type:Literal["intro","core","examples","checklist","common_mistakes","conclusion"]
    brief:str=Field(description='What to cover')

class Plan(BaseModel):
    blog_title:str
    audience:str
    tone:str
    tasks:List[Task]=Field(min_length=5,max_length=7)

class SectionResult(TypedDict):
    id:int
    content:str

class State(TypedDict):
    topic:str
    plan:Plan
    #reducer    : results from  workers get concatenated 
    sections:Annotated[List[SectionResult],operator.add]
    final:str


llm=ChatGroq(model="openai/gpt-oss-120b")

def orchestator(state:State)->dict:
    plan=llm.with_structured_output(Plan).invoke([
        SystemMessage(content=('''Create a blog plan of 5-7 sections on thr following  topic.''')
        ),HumanMessage(content=f"Topic:{state['topic']}")
    ])
    return {"plan":plan}


#intermidate function that will create  multiple workers AS per story 
def fanout(state: State):
    return [
        Send("worker", {
            "task": task,
            "topic": state['topic'],
            "plan": state['plan']
        })
        for task in state['plan'].tasks
    ]

def worker(payload:dict)->dict:
    #payload contains what we want to sent
    task=payload['task']
    topic=payload['topic']
    plan=payload['plan']

    bullets_text="\n".join(f"{index}. {bullet}" for index,bullet in enumerate(task.bullets,start=1))
    section_md=llm.invoke([
        SystemMessage(content="Write One clear markdown section"),
        HumanMessage(
            content=(f"Blog:{plan.blog_title}\n" 
                    f"Topic:{topic}\n\n"
                    f"Section:{task.title}\n"
                    f"Goal:{task.goal}\n"
                    f"Target words:{task.target_words}\n"
                    f"Bullets:{bullets_text}\n"
                    f"Brief:{task.brief}\n\n"
                    "Return only the section  content  in Markdown."),
            )
        ]
    ).content.strip()

    return{"sections":[{"id":task.id,"content":section_md}]}


#reducer code sitch the whole blog
from pathlib import Path
def reducer(state:State)->dict:
    title=state['plan'].blog_title
    ordered_sections=sorted(state['sections'],key=lambda section:section['id'])
    body="\n\n".join(section['content'] for section in ordered_sections).strip()

    final_md=f'#{title}\n\n{body}\n'


    #------save the file-----------
    filename=re.sub(r"[^A-Za-z0-9_-]+","_",title.strip()).strip("_").lower()+ ".md"
    output_path=Path(filename)
    output_path.write_text(final_md,encoding="utf-8")

    return {"final":final_md}


g=StateGraph(State)
g.add_node("orchestator",orchestator)
g.add_node("worker",worker)
g.add_node("reducer",reducer)

g.add_edge(START,"orchestator")
g.add_conditional_edges("orchestator",fanout)
g.add_edge("worker","reducer")
g.add_edge("reducer",END)

app=g.compile()

out=app.invoke({"topic":"write a blog on usuage of mobile phone increasing day by day ","sections":[]})
print(out["final"])

#The Growing Grip: How Mobile Phone Usage Is Escalating Every Day

## Introduction – A Digital Phenomenon  

The past decade has witnessed an unprecedented surge in mobile phone ownership. From bustling megacities to remote villages, smartphones now outnumber the world’s population, with **over 7 billion devices active today**—a figure that continues to climb as affordable models and expanding network coverage bring connectivity to previously unreachable corners.  

Why does this matter?  

- **Individuals** experience a reshaping of daily routines, from how they consume news to how they manage health, work, and social relationships. Understanding usage patterns helps people strike a healthier balance between screen time and offline life.  
- **Businesses** rely on mobile data to tailor marketing, optimize app experiences, and forecast demand. Real‑time insights into how, when, and where users engage with their devices drive revenue and innovation.  
- **Policymakers** need accurate us

#The Daily Surge: How Mobile Phone Usage Is Growing Every Day

## Introduction – The Ubiquity of Mobile Phones

In just a few short decades, the mobile phone has transformed from a luxury gadget into an indispensable extension of daily life. Today, a typical adult checks their device dozens of times a day—whether to send a quick text, scroll through social media, navigate a commute, or manage work tasks on the go. This constant connectivity reshapes how we communicate, consume information, and even think.

Tracking mobile‑phone usage trends isn’t just a curiosity; it’s a vital lens for understanding broader societal shifts. Rising screen time can signal changes in consumer behavior, highlight emerging opportunities (and challenges) for businesses, and reveal the impact of technology on mental health, productivity, and social interaction. By monitoring these patterns, policymakers, marketers, and individuals alike can make more informed decisions about the role of mobile technology in o

# BLog Simple but better prompting 

In [ ]:
from __future__ import annotations
import operator
from typing import TypedDict,List,Annotated,Literal
from pydantic import BaseModel,Field
from langgraph.graph import StateGraph,START,END
from langgraph.types import Send
from langchain_groq import ChatGroq
from langchain.messages import SystemMessage,HumanMessage
from dotenv import load_dotenv
load_dotenv()

#*******************Main Code*****************************************
class Task(BaseModel):
    id:int
    title:str

    goal:str=Field(
        description="One sentence  describing  what the  reader  should be  able to/do understand  after this section."
    )
    bullets:List[str]=Field(description="3-5 concrete,non overlapping subpoints  to cover  in this  section ",
                            min_length=3,
                            max_length=5,
                            )
    target_words:int=Field(description="Target word  count for  this section (120-450)")
    section_type:Literal[
        "intro","core","examples","checklist","common_mistakes","conclusion"
    ]=Field(description="use common_mistakes once  in the plan ")
    

class Plan(BaseModel):
    Blog_title:str
    audience:str=Field(description="Who is the blog for .")
    tone:str=Field(description="Writing tone (e.g.., practical,crisp)")
    tasks:List[Task]



class State(TypedDict):
    topic:str
    plan:Plan
    #reducer    : results from  workers get concatenated 
    sections:Annotated[List[str],operator.add]
    final:str


llm=ChatGroq(model="openai/gpt-oss-120b")

def orchestator(state:State)->dict:
    plan=llm.with_structured_output(Plan).invoke([
        SystemMessage(content=('''You are a senior technical writer and developer advocate. Your job is to produce a 
        "highly actionable outline for a technical blog post.\n\n"    "Hard requirements:\n"
        "- Create 5–7 sections (tasks) that fit a technical blog.\n"
        "- Each section must include:\n"
        "  1) goal (1 sentence: what the reader can do/understand after the section)\n"
        "  2) 3–5 bullets that are concrete, specific, and non-overlapping\n"
         "  3) target word count (120–450)\n"
        "- Include EXACTLY ONE section with section_type='common_mistakes'.\n\n"
        "Make it technical (not generic):\n"
        "- Assume the reader is a developer; use correct terminology.\n"
        "- Prefer design/engineering structure: problem → intuition → approach → implementation → "
        "trade-offs → testing/observability → conclusion.\n"
        "- Bullets must be actionable and testable (e.g., 'Show a minimal code snippet for X', "
        "'Explain why Y fails under Z condition', 'Add a checklist for production readiness').\n"
        "- Explicitly include at least ONE of the following somewhere in the plan (as bullets):\n"
        "  * a minimal working example (MWE) or code sketch\n"
        "  * edge cases / failure modes\n"
        "  * performance/cost considerations\n"
        "  * security/privacy considerations (if relevant)\n"
        "  * debugging tips / observability (logs, metrics, traces)\n"
        "- Avoid vague bullets like 'Explain X' or 'Discuss Y'. Every bullet should state what "
        "to build/compare/measure/verify.\n\n"
        "Ordering guidance:\n"
        "- Start with a crisp intro and problem framing.\n"
        "- Build core concepts before advanced details.\n"
        "- Include one section for common mistakes and how to avoid them.\n"
        "- End with a practical summary/checklist and next steps.\n\n"
        "Output must strictly match the Plan schema.
''')
        ),HumanMessage(content=f"Topic:{state['topic']}")
    ])
    return {"plan":plan}


#intermidate function that will create  multiple workers AS per story 
def fanout(state: State):
    return [
        Send("worker", {
            "task": task,
            "topic": state['topic'],
            "plan": state['plan']
        })
        for task in state['plan'].tasks
    ]

def worker(payload:dict)->dict:
    #payload contains what we want to sent
    task=payload['task']
    topic=payload['topic']
    plan=payload['plan']
    bullets_text = "\n".join(
    f"- {bullet}" for bullet in task.bullets
)

    blog_title=plan.Blog_title
    section_md=llm.invoke([
        SystemMessage(content="You are a senior technical writer and developer advocate. Write ONE section of a technical blog post in Markdown.\n\n"
        "Hard constraints:\n"
        "- Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).\n"
        "- Stay close to the Target words (±15%).\n"
        "- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).\n\n"
        "Technical quality bar:\n"
        "- Be precise and implementation-oriented (developers should be able to apply it).\n"
        "- Prefer concrete details over abstractions: APIs, data structures, protocols, and exact terms.\n"
        "- When relevant, include at least one of:\n"
        "  * a small code snippet (minimal, correct, and idiomatic)\n"
        "  * a tiny example input/output\n"
        "  * a checklist of steps\n"
        "  * a diagram described in text (e.g., 'Flow: A -> B -> C')\n"
        "- Explain trade-offs briefly (performance, cost, complexity, reliability).\n"
        "- Call out edge cases / failure modes and what to do about them.\n"
        "- If you mention a best practice, add the 'why' in one sentence.\n\n"
        "Markdown style:\n"
        "- Start with a '##  <Section Title>' heading.\n"
        "- Use short paragraphs, bullet lists where helpful, and code fences for code.\n"
        "- Avoid fluff. Avoid marketing language.\n"
        "- If you include code, keep it focused on the bullet being addressed.\n"),
        HumanMessage(
            content=(f"Blog: {plan.Blog_title}\n"
                    f"Audience: {plan.audience}\n"
                    f"Tone: {plan.tone}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Section type: {task.section_type}\n"
                    f"Goal: {task.goal}\n"
                    f"Target words: {task.target_words}\n"
                    f"Bullets:{bullets_text}\n"
                    "Return only the section  content  in Markdown."),
            )
        ]
    ).content.strip()

    return{"sections":[section_md]}


#reducer code sitch the whole blog
from pathlib import Path
def reducer(state:State)->dict:
    title=state['plan'].Blog_title
    body="\n\n".join(state['sections']).strip()

    final_md=f'#{title}\n\n{body}\n'


    #------save the file-----------
    filename=title.lower().replace(" ","_")+ ".md"
    output_path=Path(filename)
    output_path.write_text(final_md,encoding="utf-8")

    return {"final":final_md}


g=StateGraph(State)
g.add_node("orchestator",orchestator)
g.add_node("worker",worker)
g.add_node("reducer",reducer)

g.add_edge(START,"orchestator")
g.add_conditional_edges("orchestator",fanout)
g.add_edge("worker","reducer")
g.add_edge("reducer",END)

app=g.compile()

out=app.invoke({"topic":"Transformer ","sections":[]})